# 2. Azure ML Fine-Tuning and Model Registration

Submit reproducible QLoRA fine-tuning as an Azure ML command job, track the experiment with MLflow, persist outputs in the workspace datastore, and register an immutable candidate model.

## Experimental design

The notebook is the control plane; `lib/train.py` is the remote execution unit. Training consumes a versioned data asset, masks prompt tokens from the loss, adapts attention and MLP projections with rank-stabilized QLoRA-compatible settings, evaluates against the validation split, and writes a merged model to a named job output.

**Mandatory production controls**
- Use a dedicated GPU compute cluster with managed identity and minimum nodes set to zero.
- Pin the curated environment and data/model asset versions.
- Place gated-model credentials in Key Vault and grant the job identity only `Key Vault Secrets User`.
- Keep the test split sealed until Notebook 3. Registration here creates a candidate, not an automatic production promotion.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import sys

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "lib").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "lib").exists():
    raise RuntimeError("Start this notebook from the repository or notebooks directory")
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

from lib.azureml_ops import (
    create_training_environment,
    register_job_model,
    submit_finetuning_job,
)
from lib.config import AzureMLConfig

## Run configuration

Replace immutable asset versions deliberately. The Key Vault fields are names, not secret values. Leave both as `None` only when the base model is public or already cached.

In [ ]:
DATA_ASSET_NAME = "raft-instance-security"
DATA_ASSET_VERSION = "REPLACE_WITH_NOTEBOOK_1_VERSION"
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
COMPUTE_NAME = "gpu-a10-cluster"
TRAIN_ENVIRONMENT_NAME = "raft-qlora-train"
TRAIN_ENVIRONMENT_VERSION = "1"
EXPERIMENT_NAME = "raft-llama32-finetuning"
REGISTERED_MODEL_NAME = "raft-llama32-1b"
REGISTERED_MODEL_VERSION = datetime.now(timezone.utc).strftime("%Y%m%d.%H%M%S")
KEY_VAULT_NAME = None
HF_TOKEN_SECRET_NAME = None

assert "REPLACE" not in DATA_ASSET_VERSION

## Connect and resolve immutable inputs

`DefaultAzureCredential` uses your Azure CLI identity locally and managed identity in Azure ML. No subscription IDs or credentials are stored in notebook output.

In [ ]:
config = AzureMLConfig.from_env()
ml_client = config.create_ml_client()
data_asset = ml_client.data.get(DATA_ASSET_NAME, version=DATA_ASSET_VERSION)
print("Data input:", data_asset.id)
print("Compute:", ml_client.compute.get(COMPUTE_NAME).name)

## Register the pinned training environment

The environment definition lives in `environments/train-conda.yml`. Increment its version whenever a dependency changes; never mutate an environment used by a completed experiment.

In [ ]:
training_environment = create_training_environment(
    ml_client, TRAIN_ENVIRONMENT_NAME, TRAIN_ENVIRONMENT_VERSION
)
print("Environment:", training_environment.id)

## Submit and observe the remote job

Azure ML snapshots source code and mounts the output on workspace storage. Transformers metrics, parameters, system metrics, validation loss, and perplexity are sent to the workspace MLflow tracking server.

In [ ]:
submitted_job = submit_finetuning_job(
    ml_client=ml_client,
    data_asset=f"azureml:{DATA_ASSET_NAME}:{DATA_ASSET_VERSION}",
    environment=training_environment.id,
    compute=COMPUTE_NAME,
    experiment_name=EXPERIMENT_NAME,
    base_model=BASE_MODEL,
    display_name=f"raft-qlora-{REGISTERED_MODEL_VERSION}",
    key_vault_name=KEY_VAULT_NAME,
    hf_token_secret_name=HF_TOKEN_SECRET_NAME,
)
print("Job name:", submitted_job.name)
ml_client.jobs.stream(submitted_job.name)

## Inspect MLflow lineage

Use the experiment view for learning curves and system utilization. Programmatic lookup below makes the association between Azure ML job and MLflow run explicit.

In [ ]:
import mlflow

workspace = ml_client.workspaces.get(config.workspace_name)
mlflow.set_tracking_uri(workspace.mlflow_tracking_uri)
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    max_results=10,
    order_by=["start_time DESC"],
)
visible_columns = [
    column
    for column in runs.columns
    if column in {"run_id", "status"}
    or column.startswith("metrics.")
    or column.startswith("params.")
]
runs[visible_columns].head()

## Register the candidate model

Registration points directly to the named job output in Azure Storage, retaining job lineage without a local download/re-upload. The candidate remains unpromoted until held-out evaluation and deployment smoke tests pass in Notebook 3.

In [ ]:
completed_job = ml_client.jobs.get(submitted_job.name)
if completed_job.status != "Completed":
    raise RuntimeError(f"Training did not complete successfully: {completed_job.status}")

registered_model = register_job_model(
    ml_client=ml_client,
    job_name=submitted_job.name,
    model_name=REGISTERED_MODEL_NAME,
    version=REGISTERED_MODEL_VERSION,
)
print("Registered candidate:", registered_model.id)

## Handoff and approval evidence

Retain the data fingerprint, source commit, environment version, job name, MLflow run, model version, training/evaluation curves, and responsible approver. Cost, license acceptance, PII review, red-team results, and rollback ownership belong in the model card before production promotion.